In [1]:
# @title user defined variables

# @markdown ###**The name of the secret where the api key is stored**
api_key_secret_name = 'GOOGLE_API_KEY' #@param {type: 'string'}
# @markdown ###**The model to be called**
model_name = 'gemini-2.5-flash' #@param ["gemini-2.5-pro", "gemini-2.5-flash", "gemini-2.0-pro", "gemini-2.0-flash"]
# @markdown ###**The number of groups to be generated**
num_generated_groups = 50 #@param {type:"slider", min:50, max:50000, step:50}
# @markdown ###**The maximal amount of model calls (gemini) that are permitted**
max_model_calls = 3 #@param {type:"slider", min:1, max:1000, step:1}

In [2]:
# @title imports
# For the dataset
import kagglehub
# For general data shenenigans
import pandas as pd
import numpy as np
# For gemini :D
from google import genai
# To get the api key
from google.colab import userdata
# To parse model response
import json
# to save stuff in files
import csv
from google.colab import files

In [3]:
# @title global variables
# The name of the kaggle dataset
connection_dataset_name ="eric27n/the-new-york-times-connections"
# The name of the data file for connections
connection_csv_name = "Connections_Data.csv"
# example of a group appearing in the dataset
connections_group_example = ['LEVEL', 'KAYAK', 'RACECAR', 'MOM']
# example of a group NOT appearing in the dataset
connections_group_antiexample = ['DSAE', 'SED', 'ASD', 'AER']

# Threshold of group overlap with any group in the dataset to consider it a "leak"
group_leak_threshold = 0.75

# prompt to generate synthetic data
synthetic_data_prompt = """Role: You are an expert puzzle constructor for a game identical to New York Times "Connections".
Objective: Generate 10 complete, distinct games. Each game must consist of exactly 4 groups of four words.

Core Design Principles
* Unambiguity is Paramount: The connection must be tight. All four words must definitively belong to the category. Avoid "sort of" fits.
* No "Lazy" Synonyms: Do not just pick four words that broadly mean the same thing (e.g., BIG, HUGE, LARGE, GIANT). Good synonym categories use common words in specific contexts (e.g., "Slang for Toilet": JOHN, CAN, HEAD, THRONE).
* Embrace Red Herrings: The best puzzles use words that seem like they belong to other categories within the same game.
* Distinct Games: Ensure no categories or words overlap between the 10 generated games.

Required Difficulty Tiers Per Game
Every single game must contain exactly one group from each of the four established difficulty levels:

* 🟨 Level 1 (Straightforward): Simple semantic categories. (e.g., [ADIDAS, NIKE, PUMA, REEBOK] - Sportswear)
* 🟩 Level 2 (Medium): Slightly more specific knowledge, or common words used in a unique context. (e.g., [SHOOT, DARN, FUDGE, HECK] - Polite exclamations)
* 🟦 Level 3 (Specific Knowledge): Categories requiring trivia, specific cultural knowledge, or abstract groupings. (e.g., [WARNER, JONAS, MARX, WRIGHT] - Famous brothers)
* 🟪 Level 4 (Tricky/Wordplay): The hardest tier. NOT based on definitions. Use lateral thinking like:
    * Fill-in-the-Blank: (e.g., "___ TAPE" -> [RED, SCOTCH, DUCT, VIDEO])
    * Word Structure: (e.g., Palindromes -> [KAYAK, MOM, RACECAR, LEVEL])
    * Homophones: (e.g., Sounds like letters -> [ARE, SEA, WHY, QUEUE])
    * Hidden Words: (e.g., Animals hidden inside words -> [SCOWL, BULLY, GOATEE, RAMPANT])

Output Format:
You must output the final result EXCLUSIVELY as a raw JSON list of 10 game objects. Do not include any conversational text, markdown code block fences, or introductions before OR after the JSON.

Use this exact schema:
[
  {
    "game_id": 1,
    "groups": [
      {
        "level": 1,
        "emoji": "🟨",
        "theme": "Theme explanation here",
        "words": ["WORD1", "WORD2", "WORD3", "WORD4"]
      },
      {
        "level": 2,
        "emoji": "🟩",
        "theme": "Theme explanation here",
        "words": ["WORD1", "WORD2", "WORD3", "WORD4"]
      },
      {
        "level": 3,
        "emoji": "🟦",
        "theme": "Theme explanation here",
        "words": ["WORD1", "WORD2", "WORD3", "WORD4"]
      },
      {
        "level": 4,
        "emoji": "🟪",
        "theme": "Tricky wordplay theme here",
        "words": ["WORD1", "WORD2", "WORD3", "WORD4"]
      }
    ]
  },
  {
    "game_id": 2,
    "groups": [
      // ... groups for game 2
    ]
  }
  // ... repeat until game_id 10
]"""


api_key = userdata.get(api_key_secret_name)
client = genai.Client(api_key=api_key)

In [4]:
# @title util functions

# Downloads the dataset and returns the path of the folder
def download_connections_data(dataset_name):
  return kagglehub.dataset_download(dataset_name)

# downloads the dataset with name dataset_name and returns the file with name csv_name as pd df
def download_and_read_csv(dataset_name, csv_name):
  return pd.read_csv(download_connections_data(dataset_name)+"/"+csv_name)

# helper function to detect groups already in the dataset
# returns a rate 0<x<1, where x is the max percentage of words
# group is 1d array, word_groups is 2d array
def highest_match_rate(single_group, word_groups):
  highest_score = 0.0
  grp_set = set(single_group)
  grp_len = len(single_group)
  for word_group in word_groups:
    cur_count = len(grp_set.intersection(word_group))
    cur_score = cur_count/grp_len
    highest_score = max(highest_score, cur_score)
  return highest_score

def filter_out_leaks(groups, dataset, group_leak_threshold):
  num_removed_groups = 0
  for group in groups:
    if(highest_match_rate(group, dataset) >= group_leak_threshold):
      groups.remove(group)
      num_removed_groups += 1
  return groups, num_removed_groups


# calls the corresponding model with the corresponding name, returns the model response
def call_gemini(prompt, model_name="gemini-2.0-flash"):
  try:
    response = client.models.generate_content(model=model_name, contents=prompt)
    return response.text
  except Exception as e:
    print(f"Error: {e}")

# tries to parse a model response, the response is expected to be a 2d array
def parse_model_response(model_response):
  try:
    if isinstance(model_response, list):
      parsed_repsonse = model_response
    else:
      model_response = str(model_response)
      start_idx = model_response.find("[")
      end_idx = model_response.rfind("]")
      model_response = model_response[start_idx:end_idx+1]
      parsed_response = json.loads(model_response)
    word_groups = []
    for game in parsed_response:
      for group in game['groups']:
        word_groups.append(group['words'])
    return word_groups
  except Exception as e:
    print(f"Error: {e} with model_response {model_response}")
    return None

In [5]:
# @title load the dataframe and save the groups
df = download_and_read_csv(connection_dataset_name, connection_csv_name)
dataset_groups = df.groupby(['Game ID', 'Group Name'])['Word'].apply(set).tolist()
assert(highest_match_rate(connections_group_example, dataset_groups)==1.0)
assert(highest_match_rate(connections_group_antiexample, dataset_groups)==0.0)

100%|██████████| 146k/146k [00:00<00:00, 50.9MB/s]

Extracting files...


In [7]:
# @title call gemini to create groups, parse them and filter out leaks until we have enough examples or reached max_model_calls
remaining_model_calls = max_model_calls
generated_groups = []
while(remaining_model_calls > 0 and len(generated_groups) < num_generated_groups):
  model_response = call_gemini(synthetic_data_prompt)
  remaining_model_calls -= 1
  parsed_groups = parse_model_response(model_response)
  if(parsed_groups is None):
    continue
  filtered_groups, removed_groups = filter_out_leaks(parsed_groups, dataset_groups, group_leak_threshold)
  print(f"filtered out {removed_groups} examples and left {len(filtered_groups)}")
  # append the filtered_groups 2d array to generated_groups (also 2d array, maybe empty)
  if len(generated_groups) == 0:
    generated_groups = filtered_groups
  else:
    generated_groups.extend(filtered_groups)
  print(f"in total now generated {len(generated_groups)} groups")


filtered out 5 examples and left 35
in total now generated 35 groups
filtered out 3 examples and left 37
in total now generated 72 groups


In [8]:
# @title save the generated groups
file_name = "generated_groups.csv"
with open(file_name, "w", newline="", encoding="utf-8") as f:
  writer = csv.writer(f)
  writer.writerow(["Word 1","Word 2","Word 3","Word 4"])
  writer.writerows(generated_groups)
files.download(file_name)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>